In [1]:
import os
import time
import re
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests


In [4]:
# ----------------------------
# CONFIG
# ----------------------------
INPUT_PATH = "Instagram/ig_user_info.csv"   # or "users_details.csv"
SHEET_NAME = None                  # e.g., "Sheet1" if Excel; None = first sheet
OUTPUT_PATH = "Instagram/ig_user_info_with_profile_pic_status.csv"

OUT_DIR = Path("Instagram") / "profile pics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_ID_COL = "user_id"
URL_COL = "profile_pic_url_hd"

TIMEOUT = 30
MAX_RETRIES = 3
SLEEP_BETWEEN_REQUESTS_SEC = 0.05
USER_AGENT = "Mozilla/5.0 (compatible; ProfilePicDownloader/1.0)"

# If a cell contains multiple URLs, we take the first valid http(s) URL
url_regex = re.compile(r"https?://\S+")

def pick_first_url(val) -> str | None:
    if val is None:
        return None
    s = str(val).strip()
    if not s or s.lower() in {"nan", "none", "-"}:
        return None
    m = url_regex.search(s)
    return m.group(0) if m else None

def safe_ext_from_url(url: str) -> str:
    try:
        path = urlparse(url).path.lower()
        for ext in (".jpg", ".jpeg", ".png", ".webp"):
            if path.endswith(ext):
                return ext
    except Exception:
        pass
    return ".jpg"

def download_file(url: str, dest_path: Path) -> tuple[bool, str]:
    headers = {"User-Agent": USER_AGENT, "Accept": "*/*"}
    last_err = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=TIMEOUT, headers=headers) as r:
                if r.status_code != 200:
                    return False, f"HTTP {r.status_code}"

                ctype = (r.headers.get("Content-Type") or "").lower()
                if "image" not in ctype and "octet-stream" not in ctype:
                    return False, f"Unexpected Content-Type: {ctype or 'missing'}"

                tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 128):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp_path, dest_path)
                return True, ""
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
            time.sleep(0.4 * attempt)
    return False, last_err or "Unknown error"

# ----------------------------
# LOAD
# ----------------------------
df = pd.read_csv(INPUT_PATH)

# Validate required columns
missing = [c for c in [USER_ID_COL, URL_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {list(df.columns)}")

# Ensure output columns exist
for col in ["profile_pic_title", "result", "error_message", "profile_pic_scrape_ts"]:
    if col not in df.columns:
        df[col] = ""

In [6]:

# ----------------------------
# DOWNLOAD LOOP
# ----------------------------
for i, row in df.iterrows():
    uid = row.get(USER_ID_COL)
    title = "" if pd.isna(uid) else str(uid).strip()

    df.at[i, "profile_pic_title"] = title
    df.at[i, "profile_pic_scrape_ts"] = int(time.time())

    url = pick_first_url(row.get(URL_COL))

    if not title:
        df.at[i, "result"] = "failed"
        df.at[i, "error_message"] = f"Missing {USER_ID_COL}"
        continue

    if not url:
        df.at[i, "result"] = "failed"
        df.at[i, "error_message"] = f"Missing/invalid {URL_COL}"
        continue

    ext = safe_ext_from_url(url)
    dest = OUT_DIR / f"{title}{ext}"

    # Resume-safe
    if dest.exists() and dest.stat().st_size > 0:
        df.at[i, "result"] = "success"
        df.at[i, "error_message"] = ""
        continue

    ok, err = download_file(url, dest)
    df.at[i, "result"] = "success" if ok else "failed"
    df.at[i, "error_message"] = err
    print("downloading:", df.loc[i, "brand"], df.loc[i, "result"])


    time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

# ----------------------------
# SAVE
# ----------------------------
df.to_csv(OUTPUT_PATH, index=False)
print(f"Done.\nSaved updated table: {OUTPUT_PATH}\nSaved images to: {OUT_DIR.resolve()}")


downloading: Allianz failed
downloading: Amazon Web Services failed
downloading: Aramco failed
downloading: Deloitte failed
downloading: Deutsche Telekom failed
downloading: Google failed
downloading: Google failed
downloading: Google failed
downloading: Instagram failed
downloading: J.P. Morgan failed
downloading: Microsoft failed
downloading: Shell failed
downloading: Siemens failed
downloading: Telekom / T-Mobile failed
downloading: UnitedHealthcare failed
downloading: adidas failed
downloading: Nike failed
downloading: BMW failed
Done.
Saved updated table: Instagram/ig_user_info_with_profile_pic_status.csv
Saved images to: /Users/pouriakhansari/Library/CloudStorage/OneDrive-IveyBusinessSchool/Prashant/Content Format/Code/EnsembleData/Instagram/profile pics
